# BIM2GRAPH mejorado para accesibilidad y movilidad en IFC

Este notebook reestructura tu script original para ejecutarlo en **Google Colab** con una lógica más sólida y más útil para análisis de movilidad.

## Cambios implementados

1. **Ascensores sin etiquetas de texto**  
   Se dejan de buscar cadenas como `ASCENSOR` o `ELEVADOR` en `Name/ObjectType`.  
   Ahora se usan:
   - `IfcTransportElement`
   - `PredefinedType` cuando existe
   - criterios geométricos
   - presencia en varios niveles
   - proximidad a puertas por planta

2. **Tramos ortogonales puerta ↔ espacio**  
   La conexión ya no es una línea directa simplista.  
   Se calcula la orientación del **muro anfitrión de la puerta** y el tramo sale **perpendicular al muro**, formando un recorrido en ángulo recto hasta el nodo destino.

3. **Detección de rampas**  
   Se incorporan `IfcRamp` e `IfcRampFlight` como conectores verticales accesibles.

4. **Planta base ligera por nivel**  
   Se dibujan en gris las líneas base de:
   - muros externos
   - separadores de espacios  
   Esto hace que la visualización sea mucho más legible.

## Flujo de trabajo

1. Instalar dependencias  
2. Subir o seleccionar el archivo IFC  
3. Definir la clase principal  
4. Ejecutar la extracción y exportar el GeoPackage  
5. Revisar el resumen  
6. Visualizar el grafo en 3D y descargar el resultado


## 1. Instalar dependencias

Esta celda instala las librerías necesarias para ejecutar el flujo completo en Colab:

- `ifcopenshell` para leer el IFC
- `geopandas`, `shapely`, `fiona`, `pyogrio` para exportación y lectura de capas espaciales
- `networkx` para el grafo
- `plotly` para la visualización 3D


In [8]:
# Dependencias necesarias para trabajar en Google Colab
!pip -q install ifcopenshell geopandas shapely fiona pyogrio plotly networkx


## 2. Cargar el modelo IFC

Esta celda permite dos modos:

- **Subir un archivo IFC desde tu equipo**
- **Indicar una ruta manual** si el archivo ya está en Colab o en Google Drive

Por defecto, el notebook queda preparado para subir el archivo desde tu ordenador.


In [9]:
from google.colab import files
import os

# Opción A: subir un IFC desde tu equipo
MODO_SUBIDA = True

# Opción B: indicar una ruta ya existente en /content o en Google Drive
IFC_PATH = "/content/EPM_ModeloAccesibilidad_01_EPM.ifc"  # Ejemplo: "/content/EPM_ModeloAccesibilidad_01_EPM.ifc"

if MODO_SUBIDA:
    uploaded = files.upload()
    ifc_files = [name for name in uploaded.keys() if name.lower().endswith(".ifc")]
    if not ifc_files:
        raise ValueError("No se ha subido ningún archivo IFC.")
    IFC_PATH = ifc_files[0]
else:
    if IFC_PATH is None or not os.path.exists(IFC_PATH):
        raise FileNotFoundError("Define IFC_PATH con una ruta válida al archivo IFC.")

print(f"IFC seleccionado: {IFC_PATH}")


Saving ModeloAccesibilidad_01_EPM_solo_escalera.ifc to ModeloAccesibilidad_01_EPM_solo_escalera (2).ifc
IFC seleccionado: ModeloAccesibilidad_01_EPM_solo_escalera (2).ifc


## 3. Clase principal `BIMAccessibilityMapper`

Esta es la parte central del notebook. Aquí se implementan todas las mejoras:

- lectura geométrica del IFC
- nodos de espacios, suelos, puertas, escaleras, rampas y ascensores
- detección robusta de ascensores
- trazado ortogonal puerta ↔ espacio
- extracción de planta base por nivel
- exportación a GeoPackage

No hay magia. Hay heurísticas geométricas razonables. Y eso es justo lo que necesitas en un IFC real cuando la calidad semántica del modelo no es perfecta.


In [10]:

from collections import defaultdict
import math
import os

import geopandas as gpd
import ifcopenshell
import ifcopenshell.geom
import ifcopenshell.util.element as ifc_element
import networkx as nx
from shapely.geometry import LineString, Point

class BIMAccessibilityMapper:
    """
    Extrae un grafo de movilidad/accesibilidad a partir de un IFC.

    Cambios principales frente a la versión original:
    1) Ascensores: detección semántica+geométrica, sin depender de textos como
       "ASCENSOR" o "ELEVADOR" en Name/ObjectType.
    2) Tramos puerta-espacio y puerta-conector vertical: polilíneas ortogonales
       basadas en la orientación real del muro anfitrión de la puerta.
    3) Rampas: detección explícita de IfcRamp e IfcRampFlight.
    4) Visualización: extracción de una planta base ligera por nivel a partir de
       muros externos y separadores de espacios.
    """

    def __init__(self, ifc_path):
        self.ifc_path = ifc_path
        self.model = ifcopenshell.open(ifc_path)
        self.G = nx.Graph()
        self.filename = os.path.splitext(os.path.basename(ifc_path))[0]

        self.settings = ifcopenshell.geom.settings()
        self.settings.set(self.settings.USE_WORLD_COORDS, True)

        self.storey_by_name = {}
        self.storey_elev_by_name = {}

        for st in self.model.by_type("IfcBuildingStorey"):
            name = (st.Name or "Nivel Desconocido").strip()
            self.storey_by_name[name] = st
            elev = getattr(st, "Elevation", None)
            try:
                elev_val = float(elev) if elev is not None else 0.0
            except Exception:
                elev_val = 0.0
            self.storey_elev_by_name[name] = elev_val

        if not self.storey_elev_by_name:
            print("Advertencia: no se encontraron IfcBuildingStorey. Se asume un nivel base en Z=0.0.")
            self.storey_elev_by_name["Nivel 0"] = 0.0

        self._sorted_storeys = sorted(self.storey_elev_by_name.items(), key=lambda kv: kv[1])

        self._geom_cache = {}
        self._bbox2d_by_space = {}
        self._space_centers = {}
        self._doors_by_level = defaultdict(list)
        self._storey_polylines = defaultdict(list)

    # -------------------------------------------------------------------------
    # Utilidades geométricas
    # -------------------------------------------------------------------------
    def get_element_geometry(self, element):
        gid = getattr(element, "GlobalId", id(element))
        if gid in self._geom_cache:
            return self._geom_cache[gid]

        try:
            shape = ifcopenshell.geom.create_shape(self.settings, element)
            verts = shape.geometry.verts
            xs = verts[0::3]
            ys = verts[1::3]
            zs = verts[2::3]

            if not xs:
                self._geom_cache[gid] = (None, None)
                return None, None

            centroid = (sum(xs) / len(xs), sum(ys) / len(ys), sum(zs) / len(zs))
            bbox = (min(xs), min(ys), min(zs), max(xs), max(ys), max(zs))
            self._geom_cache[gid] = (centroid, bbox)
            return centroid, bbox
        except Exception:
            self._geom_cache[gid] = (None, None)
            return None, None

    def get_element_vertices(self, element):
        try:
            shape = ifcopenshell.geom.create_shape(self.settings, element)
            verts = shape.geometry.verts
            return list(zip(verts[0::3], verts[1::3], verts[2::3]))
        except Exception:
            return []

    def snap_z_to_level(self, z_value):
        try:
            z = float(z_value)
        except Exception:
            z = 0.0

        closest_name, closest_elev = min(self._sorted_storeys, key=lambda kv: abs(kv[1] - z))
        return closest_name, float(closest_elev)

    def check_proximity(self, point, bbox, tolerance=0.5):
        px, py, pz = point
        min_x, min_y, min_z, max_x, max_y, max_z = bbox
        return ((min_x - tolerance <= px <= max_x + tolerance) and
                (min_y - tolerance <= py <= max_y + tolerance) and
                (min_z - tolerance <= pz <= max_z + tolerance))

    def distance_2d(self, a, b):
        return math.hypot(a[0] - b[0], a[1] - b[1])

    def bbox2d_contains(self, xy, bbox2d, tolerance=0.3):
        x, y = xy
        min_x, min_y, max_x, max_y = bbox2d
        return (min_x - tolerance <= x <= max_x + tolerance and
                min_y - tolerance <= y <= max_y + tolerance)

    def bbox2d_distance(self, xy, bbox2d):
        x, y = xy
        min_x, min_y, max_x, max_y = bbox2d
        dx = max(min_x - x, 0, x - max_x)
        dy = max(min_y - y, 0, y - max_y)
        return math.hypot(dx, dy)

    def element_storey_name(self, element):
        try:
            container = ifc_element.get_container(element)
            if container and container.is_a("IfcBuildingStorey"):
                return (container.Name or "Nivel Desconocido").strip()
        except Exception:
            pass

        centroid, bbox = self.get_element_geometry(element)
        if bbox:
            return self.snap_z_to_level(bbox[2])[0]
        if centroid:
            return self.snap_z_to_level(centroid[2])[0]
        return "Nivel Desconocido"

    # -------------------------------------------------------------------------
    # Utilidades vectoriales para trayectorias ortogonales respecto al muro
    # -------------------------------------------------------------------------
    def normalize_2d(self, vec):
        x, y = vec
        norm = math.hypot(x, y)
        if norm < 1e-9:
            return None
        return (x / norm, y / norm)

    def principal_axis_2d(self, points_xy):
        # PCA 2D mínima sin depender de numpy
        if len(points_xy) < 2:
            return None

        mx = sum(p[0] for p in points_xy) / len(points_xy)
        my = sum(p[1] for p in points_xy) / len(points_xy)

        sxx = sum((p[0] - mx) ** 2 for p in points_xy)
        syy = sum((p[1] - my) ** 2 for p in points_xy)
        sxy = sum((p[0] - mx) * (p[1] - my) for p in points_xy)

        # Autovector principal de la matriz [[sxx, sxy], [sxy, syy]]
        trace = sxx + syy
        det = sxx * syy - sxy * sxy
        disc = max(trace * trace / 4.0 - det, 0.0)
        eig = trace / 2.0 + math.sqrt(disc)

        vx = sxy
        vy = eig - sxx

        if abs(vx) < 1e-9 and abs(vy) < 1e-9:
            # fallback simple
            vx, vy = (1.0, 0.0) if sxx >= syy else (0.0, 1.0)

        return self.normalize_2d((vx, vy))

    def get_host_wall_for_door(self, door):
        # Ruta IFC típica: IfcDoor -> IfcRelFillsElement -> IfcOpeningElement
        # -> IfcRelVoidsElement -> muro anfitrión
        try:
            for rel_fill in self.model.by_type("IfcRelFillsElement"):
                related = getattr(rel_fill, "RelatedBuildingElement", None)
                if related and getattr(related, "GlobalId", None) == getattr(door, "GlobalId", None):
                    opening = getattr(rel_fill, "RelatingOpeningElement", None)
                    if opening is None:
                        continue
                    for rel_void in self.model.by_type("IfcRelVoidsElement"):
                        related_opening = getattr(rel_void, "RelatedOpeningElement", None)
                        if related_opening and getattr(related_opening, "GlobalId", None) == getattr(opening, "GlobalId", None):
                            wall = getattr(rel_void, "RelatingBuildingElement", None)
                            if wall and wall.is_a() in ("IfcWall", "IfcWallStandardCase", "IfcCurtainWall"):
                                return wall
        except Exception:
            pass
        return None

    def infer_wall_axis_2d(self, wall_or_door):
        verts = self.get_element_vertices(wall_or_door)
        if len(verts) < 2:
            return (1.0, 0.0)

        zs = [v[2] for v in verts]
        min_z = min(zs)
        pts_xy = [(x, y) for x, y, z in verts if z <= min_z + 0.40]
        if len(pts_xy) < 2:
            pts_xy = [(x, y) for x, y, _ in verts]

        axis = self.principal_axis_2d(pts_xy)
        return axis or (1.0, 0.0)

    def infer_door_frame(self, door):
        wall = self.get_host_wall_for_door(door)
        wall_axis = self.infer_wall_axis_2d(wall if wall is not None else door)
        wall_normal = (-wall_axis[1], wall_axis[0])
        return wall_axis, wall_normal

    def orthogonal_path_from_door(self, door_xy, target_xy, wall_axis, wall_normal):
        # La ruta sigue exactamente tu criterio:
        # 1) sale desde la puerta en dirección perpendicular al muro
        # 2) llega hasta la intersección con la recta paralela al muro que pasa por el nodo destino
        dx = target_xy[0] - door_xy[0]
        dy = target_xy[1] - door_xy[1]

        # Elegimos el sentido de la normal de forma que apunte hacia el destino
        dot_n = dx * wall_normal[0] + dy * wall_normal[1]
        nx = wall_normal[0] if dot_n >= 0 else -wall_normal[0]
        ny = wall_normal[1] if dot_n >= 0 else -wall_normal[1]

        t = dx * nx + dy * ny
        px = door_xy[0] + t * nx
        py = door_xy[1] + t * ny

        pts = [door_xy]
        if self.distance_2d(door_xy, (px, py)) > 1e-6:
            pts.append((px, py))
        if self.distance_2d((px, py), target_xy) > 1e-6:
            pts.append(target_xy)
        if len(pts) == 1:
            pts.append(target_xy)
        return pts

    def orthogonal_path_xy(self, src_xy, dst_xy, prefer_axis="x"):
        sx, sy = src_xy
        dx, dy = dst_xy
        if prefer_axis == "x":
            mid = (dx, sy)
        else:
            mid = (sx, dy)

        pts = [(sx, sy)]
        if self.distance_2d((sx, sy), mid) > 1e-6:
            pts.append(mid)
        if self.distance_2d(mid, (dx, dy)) > 1e-6:
            pts.append((dx, dy))
        if len(pts) == 1:
            pts.append((dx, dy))
        return pts

    def add_edge_with_polyline(self, u, v, coords3d, weight=1.0, accessible=True, edge_type="camino"):
        if len(coords3d) < 2:
            return

        geom = LineString(coords3d)
        levels = "|".join(sorted(set([
            str(self.G.nodes[u].get("level", "Desconocido")),
            str(self.G.nodes[v].get("level", "Desconocido")),
        ])))

        self.G.add_edge(
            u, v,
            weight=float(weight),
            accessible=bool(accessible),
            edge_type=edge_type,
            geometry=geom,
            levels=levels
        )

    def connect_door_to_space_orthogonal(self, door_id, space_info, weight, accessible):
        door_node = self.G.nodes[door_id]
        space_node = self.G.nodes[space_info["id"]]

        door_xy = (door_node["x"], door_node["y"])
        target_xy = (space_node["x"], space_node["y"])
        wall_axis = (door_node["wall_axis_x"], door_node["wall_axis_y"])
        wall_normal = (door_node["wall_normal_x"], door_node["wall_normal_y"])

        coords2d = self.orthogonal_path_from_door(door_xy, target_xy, wall_axis, wall_normal)
        coords3d = [(x, y, door_node["z"]) for x, y in coords2d]
        self.add_edge_with_polyline(
            door_id, space_info["id"], coords3d,
            weight=weight, accessible=accessible, edge_type="puerta_espacio"
        )

    def connect_vertical_to_space(self, node_id, space_info, weight=1.0, accessible=True):
        a = self.G.nodes[node_id]
        b = self.G.nodes[space_info["id"]]
        src_xy = (a["x"], a["y"])
        dst_xy = (b["x"], b["y"])

        coords2d = self.orthogonal_path_xy(src_xy, dst_xy, prefer_axis="x")
        coords3d = [(x, y, a["z"]) for x, y in coords2d]
        self.add_edge_with_polyline(
            node_id, space_info["id"], coords3d,
            weight=weight, accessible=accessible, edge_type="vertical_espacio"
        )

    def connect_vertical_to_door(self, node_id, door_id, weight=1.0, accessible=True):
        vertical_node = self.G.nodes[node_id]
        door_node = self.G.nodes[door_id]

        door_xy = (door_node["x"], door_node["y"])
        target_xy = (vertical_node["x"], vertical_node["y"])
        wall_axis = (door_node["wall_axis_x"], door_node["wall_axis_y"])
        wall_normal = (door_node["wall_normal_x"], door_node["wall_normal_y"])

        coords2d = self.orthogonal_path_from_door(door_xy, target_xy, wall_axis, wall_normal)
        coords2d = list(reversed(coords2d))
        coords3d = [(x, y, vertical_node["z"]) for x, y in coords2d]
        self.add_edge_with_polyline(
            node_id, door_id, coords3d,
            weight=weight, accessible=accessible, edge_type="vertical_puerta"
        )

    # -------------------------------------------------------------------------
    # Planta base simplificada
    # -------------------------------------------------------------------------
    def is_external_wall(self, wall):
        try:
            pset = ifc_element.get_pset(wall, "Pset_WallCommon")
            if isinstance(pset, dict):
                value = pset.get("IsExternal", None)
                if value is not None:
                    return bool(value)
        except Exception:
            pass
        return False

    def extract_low_edges(self, element, max_rel_z=0.25):
        verts = self.get_element_vertices(element)
        if not verts:
            return []

        zs = [v[2] for v in verts]
        min_z = min(zs)
        low_pts = [(x, y) for x, y, z in verts if z <= min_z + max_rel_z]
        if len(low_pts) < 2:
            return []

        # Eliminamos duplicados por tolerancia
        unique_pts = []
        seen = set()
        for x, y in low_pts:
            key = (round(x, 3), round(y, 3))
            if key not in seen:
                seen.add(key)
                unique_pts.append((x, y))

        if len(unique_pts) < 2:
            return []

        lines = []
        for i in range(len(unique_pts) - 1):
            if self.distance_2d(unique_pts[i], unique_pts[i + 1]) > 0.20:
                lines.append(LineString([unique_pts[i], unique_pts[i + 1]]))
        if len(unique_pts) > 2 and self.distance_2d(unique_pts[-1], unique_pts[0]) > 0.20:
            lines.append(LineString([unique_pts[-1], unique_pts[0]]))
        return lines

    def collect_storey_outline(self):
        wall_like = {"IfcWall", "IfcWallStandardCase", "IfcCurtainWall"}
        seen = set()

        # 1) Muros/separadores encontrados a través de límites de espacios
        for rel in self.model.by_type("IfcRelSpaceBoundary"):
            elem = getattr(rel, "RelatedBuildingElement", None)
            if elem is None or elem.is_a() not in wall_like:
                continue

            lvl = self.element_storey_name(elem)
            gid = getattr(elem, "GlobalId", None)
            key = (gid, lvl)
            if key in seen:
                continue
            seen.add(key)

            for line in self.extract_low_edges(elem):
                self._storey_polylines[lvl].append(line)

        # 2) Muros externos explícitos
        for wall in self.model.by_type("IfcWall") + self.model.by_type("IfcWallStandardCase"):
            if not self.is_external_wall(wall):
                continue

            lvl = self.element_storey_name(wall)
            gid = getattr(wall, "GlobalId", None)
            key = (gid, lvl)
            if key in seen:
                continue
            seen.add(key)

            for line in self.extract_low_edges(wall):
                self._storey_polylines[lvl].append(line)

        # 3) Fallback si el IFC no trae límites de espacios ni IsExternal
        if not self._storey_polylines:
            for wall in self.model.by_type("IfcWall") + self.model.by_type("IfcWallStandardCase"):
                lvl = self.element_storey_name(wall)
                gid = getattr(wall, "GlobalId", None)
                key = (gid, lvl)
                if key in seen:
                    continue
                seen.add(key)

                for line in self.extract_low_edges(wall):
                    self._storey_polylines[lvl].append(line)

    # -------------------------------------------------------------------------
    # Ascensores: detección semántica + geométrica, sin depender de textos libres
    # -------------------------------------------------------------------------
    def identify_lift_candidates(self):
        candidates = []

        for elem in self.model.by_type("IfcTransportElement"):
            ptype = str(getattr(elem, "PredefinedType", "") or "").upper()
            # No dependemos de Name/ObjectType. Filtramos por tipo IFC si existe,
            # y si está vacío seguimos con la validación geométrica.
            if ptype in {"ESCALATOR", "MOVINGWALKWAY", "CRANEWAY"}:
                continue
            candidates.append(elem)

        good = []
        seen = set()

        for elem in candidates:
            gid = getattr(elem, "GlobalId", None)
            if gid in seen:
                continue
            seen.add(gid)

            centroid, bbox = self.get_element_geometry(elem)
            if not bbox:
                continue

            dx = bbox[3] - bbox[0]
            dy = bbox[4] - bbox[1]
            dz = bbox[5] - bbox[2]

            touched_levels = [
                lvl for lvl, elev in self._sorted_storeys
                if bbox[2] - 0.50 <= elev <= bbox[5] + 0.50
            ]

            # Criterios geométricos básicos de cabina/hueco:
            # - huella razonable
            # - desarrollo vertical suficiente
            # - presencia en >= 2 niveles
            if not (0.80 <= dx <= 6.00 and 0.80 <= dy <= 6.00):
                continue
            if dz < 2.20:
                continue
            if len(touched_levels) < 2:
                continue

            # Validación espacial: en al menos un nivel debe existir una puerta cercana.
            door_hits = 0
            for lvl in touched_levels:
                level_doors = self._doors_by_level.get(lvl, [])
                if any(self.distance_2d((centroid[0], centroid[1]), (d["x"], d["y"])) <= 3.0 for d in level_doors):
                    door_hits += 1

            if door_hits >= 1:
                good.append((elem, centroid, bbox, touched_levels))

        return good

    # -------------------------------------------------------------------------
    # Extracción principal
    # -------------------------------------------------------------------------
    def extraer_datos(self):
        print("--- Extracción mejorada: ascensores semánticos+geométricos, rampas, tramos ortogonales y planta base ---")
        spaces_data = []

        # 1) Espacios
        print("Procesando espacios...")
        for space in self.model.by_type("IfcSpace"):
            centroid, bbox = self.get_element_geometry(space)
            if not centroid:
                continue

            level_name, floor_z = self.snap_z_to_level(bbox[2])
            bbox2d = (bbox[0], bbox[1], bbox[3], bbox[4])

            self.G.add_node(
                space.GlobalId,
                name=space.Name or "Estancia",
                type="Habitacion",
                level=level_name,
                x=float(centroid[0]), y=float(centroid[1]), z=float(floor_z),
                accessible=True
            )

            info = {
                "id": space.GlobalId,
                "bbox": bbox,
                "bbox2d": bbox2d,
                "name": space.Name,
                "level": level_name
            }
            spaces_data.append(info)
            self._bbox2d_by_space[space.GlobalId] = bbox2d
            self._space_centers[space.GlobalId] = (centroid[0], centroid[1])

        # 1.5) Losas pequeñas no cubiertas por IfcSpace
        print("Buscando suelos/pasillos (IfcSlab) no estructurales...")
        for slab in self.model.by_type("IfcSlab"):
            ptype = str(getattr(slab, "PredefinedType", "") or "").upper()
            if ptype in {"ROOF", "BASESLAB"}:
                continue

            centroid, bbox = self.get_element_geometry(slab)
            if not centroid:
                continue

            dx = bbox[3] - bbox[0]
            dy = bbox[4] - bbox[1]
            if dx > 10.0 and dy > 10.0:
                continue

            is_covered = False
            for sp in spaces_data:
                if self.check_proximity(centroid, sp["bbox"], tolerance=0.10):
                    is_covered = True
                    break

            if is_covered:
                continue

            level_name, floor_z = self.snap_z_to_level(bbox[2])
            bbox2d = (bbox[0], bbox[1], bbox[3], bbox[4])

            self.G.add_node(
                slab.GlobalId,
                name=slab.Name or "Suelo/Pasillo",
                type="Suelo",
                level=level_name,
                x=float(centroid[0]), y=float(centroid[1]), z=float(floor_z),
                accessible=True
            )

            info = {
                "id": slab.GlobalId,
                "bbox": bbox,
                "bbox2d": bbox2d,
                "name": "Suelo",
                "level": level_name
            }
            spaces_data.append(info)
            self._bbox2d_by_space[slab.GlobalId] = bbox2d
            self._space_centers[slab.GlobalId] = (centroid[0], centroid[1])

        # 2) Puertas
        print("Procesando puertas...")
        num_doors_found = 0

        for door in self.model.by_type("IfcDoor"):
            centroid, bbox = self.get_element_geometry(door)
            if not centroid:
                continue

            num_doors_found += 1
            level_name, floor_z = self.snap_z_to_level(bbox[2])

            width = float(getattr(door, "OverallWidth", 0.0) or 0.0)
            is_accessible = width >= 0.85 if width > 0 else True
            weight = 1.0 if is_accessible else 999999.0

            wall_axis, wall_normal = self.infer_door_frame(door)

            self.G.add_node(
                door.GlobalId,
                name=door.Name or "Puerta",
                type="Puerta",
                level=level_name,
                x=float(centroid[0]), y=float(centroid[1]), z=float(floor_z),
                width=width,
                accessible=is_accessible,
                wall_axis_x=float(wall_axis[0]),
                wall_axis_y=float(wall_axis[1]),
                wall_normal_x=float(wall_normal[0]),
                wall_normal_y=float(wall_normal[1]),
            )

            door_info = {
                "id": door.GlobalId,
                "x": centroid[0], "y": centroid[1], "z": floor_z,
                "level": level_name,
                "bbox": bbox,
                "wall_axis": wall_axis,
                "wall_normal": wall_normal,
            }
            self._doors_by_level[level_name].append(door_info)

            connected_spaces = []
            for space_info in spaces_data:
                if space_info["level"] != level_name:
                    continue

                if self.bbox2d_contains((centroid[0], centroid[1]), space_info["bbox2d"], tolerance=0.65):
                    connected_spaces.append((0.0, space_info))
                else:
                    d = self.bbox2d_distance((centroid[0], centroid[1]), space_info["bbox2d"])
                    if d <= 1.10:
                        connected_spaces.append((d, space_info))

            connected_spaces = [s for _, s in sorted(connected_spaces, key=lambda t: t[0])[:2]]
            for space_info in connected_spaces:
                self.connect_door_to_space_orthogonal(
                    door.GlobalId, space_info, weight=weight, accessible=is_accessible
                )

        # 3) Escaleras
        print("Procesando escaleras...")
        num_stairs_found = 0
        stair_elements = self.model.by_type("IfcStairFlight") # self.model.by_type("IfcStair") +
        processed_ids = set()

        for stair in stair_elements:
            if stair.GlobalId in processed_ids:
                continue

            verts = self.get_element_vertices(stair)
            if not verts:
                continue

            xs = [v[0] for v in verts]
            ys = [v[1] for v in verts]
            zs = [v[2] for v in verts]

            min_z, max_z = min(zs), max(zs)
            idx_min, idx_max = zs.index(min_z), zs.index(max_z)
            p_start = (xs[idx_min], ys[idx_min], zs[idx_min])
            p_end = (xs[idx_max], ys[idx_max], zs[idx_max])

            level_start, floor_z_start = self.snap_z_to_level(p_start[2])
            level_end, floor_z_end = self.snap_z_to_level(p_end[2])

            id_start = f"{stair.GlobalId}_START"
            id_end = f"{stair.GlobalId}_END"

            self.G.add_node(
                id_start, name="Escalera Inicio", type="Escalera", level=level_start,
                x=float(p_start[0]), y=float(p_start[1]), z=float(floor_z_start),
                accessible=False
            )
            self.G.add_node(
                id_end, name="Escalera Fin", type="Escalera", level=level_end,
                x=float(p_end[0]), y=float(p_end[1]), z=float(floor_z_end),
                accessible=False
            )

            self.add_edge_with_polyline(
                id_start, id_end,
                [(p_start[0], p_start[1], floor_z_start), (p_end[0], p_end[1], floor_z_end)],
                weight=999999.0, accessible=False, edge_type="escalera"
            )

            for node_id, p, lvl in [(id_start, p_start, level_start), (id_end, p_end, level_end)]:
                linked = False

                for space_info in spaces_data:
                    if space_info["level"] == lvl and self.check_proximity(
                        (p[0], p[1], self.storey_elev_by_name[lvl]),
                        space_info["bbox"], tolerance=0.35
                    ):
                        self.connect_vertical_to_space(node_id, space_info, weight=1.0, accessible=True)
                        linked = True
                        break

                if not linked:
                    same_level_doors = self._doors_by_level.get(lvl, [])
                    if same_level_doors:
                        nearest = min(same_level_doors, key=lambda d: self.distance_2d((p[0], p[1]), (d["x"], d["y"])))
                        if self.distance_2d((p[0], p[1]), (nearest["x"], nearest["y"])) <= 3.0:
                            self.connect_vertical_to_door(node_id, nearest["id"], weight=1.0, accessible=True)

            num_stairs_found += 1
            processed_ids.add(stair.GlobalId)

        # 4) Rampas
        print("Procesando rampas...")
        num_ramps_found = 0
        ramp_elements = self.model.by_type("IfcRamp") + self.model.by_type("IfcRampFlight")
        processed_ids = set()

        for ramp in ramp_elements:
            if ramp.GlobalId in processed_ids:
                continue

            verts = self.get_element_vertices(ramp)
            if not verts:
                continue

            xs = [v[0] for v in verts]
            ys = [v[1] for v in verts]
            zs = [v[2] for v in verts]

            min_z, max_z = min(zs), max(zs)
            if max_z - min_z < 0.15:
                continue

            idx_min, idx_max = zs.index(min_z), zs.index(max_z)
            p_start = (xs[idx_min], ys[idx_min], zs[idx_min])
            p_end = (xs[idx_max], ys[idx_max], zs[idx_max])

            level_start, floor_z_start = self.snap_z_to_level(p_start[2])
            level_end, floor_z_end = self.snap_z_to_level(p_end[2])

            id_start = f"{ramp.GlobalId}_START"
            id_end = f"{ramp.GlobalId}_END"

            self.G.add_node(
                id_start, name="Rampa Inicio", type="Rampa", level=level_start,
                x=float(p_start[0]), y=float(p_start[1]), z=float(floor_z_start),
                accessible=True
            )
            self.G.add_node(
                id_end, name="Rampa Fin", type="Rampa", level=level_end,
                x=float(p_end[0]), y=float(p_end[1]), z=float(floor_z_end),
                accessible=True
            )

            self.add_edge_with_polyline(
                id_start, id_end,
                [(p_start[0], p_start[1], floor_z_start), (p_end[0], p_end[1], floor_z_end)],
                weight=1.20, accessible=True, edge_type="rampa"
            )

            for node_id, p, lvl in [(id_start, p_start, level_start), (id_end, p_end, level_end)]:
                linked = False

                for space_info in spaces_data:
                    if space_info["level"] == lvl and self.check_proximity(
                        (p[0], p[1], self.storey_elev_by_name[lvl]),
                        space_info["bbox"], tolerance=0.40
                    ):
                        self.connect_vertical_to_space(node_id, space_info, weight=1.0, accessible=True)
                        linked = True
                        break

                if not linked:
                    same_level_doors = self._doors_by_level.get(lvl, [])
                    if same_level_doors:
                        nearest = min(same_level_doors, key=lambda d: self.distance_2d((p[0], p[1]), (d["x"], d["y"])))
                        if self.distance_2d((p[0], p[1]), (nearest["x"], nearest["y"])) <= 3.0:
                            self.connect_vertical_to_door(node_id, nearest["id"], weight=1.0, accessible=True)

            num_ramps_found += 1
            processed_ids.add(ramp.GlobalId)

        # 5) Ascensores
        print("Procesando ascensores...")
        num_lifts_found = 0
        lift_candidates = self.identify_lift_candidates()

        for lift, centroid, bbox, touched_levels in lift_candidates:
            num_lifts_found += 1
            prev_node_id = None

            for lvl_name in touched_levels:
                lvl_z = self.storey_elev_by_name[lvl_name]
                node_id = f"{lift.GlobalId}_{lvl_name}"

                self.G.add_node(
                    node_id,
                    name="Ascensor",
                    type="Ascensor",
                    level=lvl_name,
                    x=float(centroid[0]), y=float(centroid[1]), z=float(lvl_z),
                    accessible=True
                )

                if prev_node_id:
                    prev = self.G.nodes[prev_node_id]
                    self.add_edge_with_polyline(
                        prev_node_id, node_id,
                        [(prev["x"], prev["y"], prev["z"]), (centroid[0], centroid[1], lvl_z)],
                        weight=1.0, accessible=True, edge_type="ascensor"
                    )

                prev_node_id = node_id

                same_level_doors = self._doors_by_level.get(lvl_name, [])
                linked = False

                nearby_doors = sorted(
                    same_level_doors,
                    key=lambda d: self.distance_2d((centroid[0], centroid[1]), (d["x"], d["y"]))
                )[:2]

                for door_info in nearby_doors:
                    if self.distance_2d((centroid[0], centroid[1]), (door_info["x"], door_info["y"])) <= 3.0:
                        self.connect_vertical_to_door(node_id, door_info["id"], weight=1.0, accessible=True)
                        linked = True

                if not linked:
                    candidate_spaces = [sp for sp in spaces_data if sp["level"] == lvl_name]
                    candidate_spaces = sorted(
                        candidate_spaces,
                        key=lambda sp: self.bbox2d_distance((centroid[0], centroid[1]), sp["bbox2d"])
                    )
                    for sp in candidate_spaces[:2]:
                        if self.bbox2d_distance((centroid[0], centroid[1]), sp["bbox2d"]) <= 2.5:
                            self.connect_vertical_to_space(node_id, sp, weight=1.0, accessible=True)
                            linked = True
                            break

        # 6) Planta base
        self.collect_storey_outline()

        print(
            f"Total espacios/suelos: {len(spaces_data)} | "
            f"Puertas: {num_doors_found} | "
            f"Escaleras: {num_stairs_found} | "
            f"Rampas: {num_ramps_found} | "
            f"Ascensores: {num_lifts_found}"
        )
        print(f"Grafo final: {self.G.number_of_nodes()} nodos y {self.G.number_of_edges()} aristas.")

    # -------------------------------------------------------------------------
    # Ruta accesible
    # -------------------------------------------------------------------------
    def calcular_ruta(self, start_node_id, end_node_id):
        try:
            accessible_edges = [
                (u, v)
                for u, v, d in self.G.edges(data=True)
                if d.get("accessible", True)
            ]
            subgraph = self.G.edge_subgraph(accessible_edges).copy()
            return nx.shortest_path(subgraph, source=start_node_id, target=end_node_id, weight="weight")
        except Exception:
            return "No existe ruta accesible entre los nodos indicados."

    # -------------------------------------------------------------------------
    # Exportación a GeoPackage
    # -------------------------------------------------------------------------
    def exportar_geopackage(self):
        # El IFC suele trabajar en coordenadas locales de ingeniería, no geográficas.
        # Por eso no fijamos un EPSG geográfico por defecto.
        gpkg_name = f"{self.filename}_grafo.gpkg"

        nodes_data = []
        for n, attr in self.G.nodes(data=True):
            nodes_data.append({
                "geometry": Point(attr["x"], attr["y"], attr["z"]),
                "id": n,
                "tipo": attr["type"],
                "nombre": attr.get("name", ""),
                "level": attr.get("level", "Desconocido"),
                "acc": attr.get("accessible", True),
            })

        if nodes_data:
            gdf_nodes = gpd.GeoDataFrame(nodes_data, geometry="geometry")
            gdf_nodes.to_file(gpkg_name, layer="nodos", driver="GPKG")

        edges_data = []
        for u, v, attr in self.G.edges(data=True):
            geom = attr.get("geometry", None)
            if geom is None:
                p1 = self.G.nodes[u]
                p2 = self.G.nodes[v]
                geom = LineString([(p1["x"], p1["y"], p1["z"]), (p2["x"], p2["y"], p2["z"])])

            edges_data.append({
                "geometry": geom,
                "acc": attr.get("accessible", True),
                "levels": attr.get("levels", ""),
                "edge_type": attr.get("edge_type", "camino"),
            })

        if edges_data:
            gdf_edges = gpd.GeoDataFrame(edges_data, geometry="geometry")
            gdf_edges.to_file(gpkg_name, layer="caminos", driver="GPKG")

        outline_data = []
        for lvl, lines in self._storey_polylines.items():
            z = self.storey_elev_by_name.get(lvl, 0.0)
            for line in lines:
                coords3d = [(x, y, z) for x, y in line.coords]
                outline_data.append({
                    "geometry": LineString(coords3d),
                    "level": lvl,
                    "tipo": "PlantaBase",
                })

        if outline_data:
            gdf_outline = gpd.GeoDataFrame(outline_data, geometry="geometry")
            gdf_outline.to_file(gpkg_name, layer="plantas_base", driver="GPKG")

        print(f"GeoPackage exportado: {gpkg_name}")
        return gpkg_name


## 4. Ejecutar la extracción y exportar el GeoPackage

Esta celda:

1. crea el objeto `mapper`
2. recorre el IFC
3. construye el grafo
4. exporta el resultado a un archivo `.gpkg`

Ese GeoPackage contiene tres capas:

- `nodos`
- `caminos`
- `plantas_base`


In [11]:
# Ejecutar la extracción y exportar el resultado a GeoPackage
mapper = BIMAccessibilityMapper(IFC_PATH)
mapper.extraer_datos()
gpkg_filename = mapper.exportar_geopackage()

print(f"Archivo generado: {gpkg_filename}")
print(f"Nodos: {mapper.G.number_of_nodes()} | Aristas: {mapper.G.number_of_edges()}")


--- Extracción mejorada: ascensores semánticos+geométricos, rampas, tramos ortogonales y planta base ---
Procesando espacios...
Buscando suelos/pasillos (IfcSlab) no estructurales...
Procesando puertas...
Procesando escaleras...
Procesando rampas...
Procesando ascensores...
Total espacios/suelos: 13 | Puertas: 6 | Escaleras: 10 | Rampas: 0 | Ascensores: 0
Grafo final: 39 nodos y 42 aristas.
GeoPackage exportado: ModeloAccesibilidad_01_EPM_solo_escalera (2)_grafo.gpkg
Archivo generado: ModeloAccesibilidad_01_EPM_solo_escalera (2)_grafo.gpkg
Nodos: 39 | Aristas: 42


/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:917: UserWarning:

'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.

/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:917: UserWarning:

'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.

/usr/local/lib/python3.12/dist-packages/pyogrio/geopandas.py:917: UserWarning:

'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.



## 5. Revisar un resumen rápido del resultado

Esta celda sirve para validar rápidamente si el grafo tiene sentido antes de pasar a la visualización:

- cuántos nodos hay de cada tipo
- cuántos elementos se han detectado por nivel
- una muestra de nodos con atributos


In [12]:
from collections import Counter

tipos = Counter(nx.get_node_attributes(mapper.G, "type").values())
niveles = Counter(nx.get_node_attributes(mapper.G, "level").values())

print("Resumen por tipo de nodo:")
for k, v in tipos.items():
    print(f"  - {k}: {v}")

print("\nResumen por nivel:")
for k, v in niveles.items():
    print(f"  - {k}: {v}")

print("\nPrimeros 10 nodos:")
for i, (node_id, data) in enumerate(mapper.G.nodes(data=True)):
    if i >= 10:
        break
    print(node_id, data)


Resumen por tipo de nodo:
  - Habitacion: 6
  - Suelo: 7
  - Puerta: 6
  - Escalera: 20

Resumen por nivel:
  - Nivel 3: 7
  - Nivel 1: 7
  - Nivel -1: 6
  - Nivel 0: 7
  - Nivel 2: 7
  - Nivel 4: 4
  - Nivel 5: 1

Primeros 10 nodos:
3Yxs4ustHAdPNvWSWnigZo {'name': '26', 'type': 'Habitacion', 'level': 'Nivel 3', 'x': 18.07338554281733, 'y': 2.9181818181098493, 'z': 14.099999999999989, 'accessible': True}
1GEH8_Nlv8ExAELiqhI5_3 {'name': '72', 'type': 'Habitacion', 'level': 'Nivel 1', 'x': 17.65443147255472, 'y': 2.67201102660512, 'z': 4.7, 'accessible': True}
1ifIJNTgHBU9_Mymb8zSse {'name': '115', 'type': 'Habitacion', 'level': 'Nivel -1', 'x': 18.573116303505955, 'y': 2.889999999999757, 'z': -4.000000000000003, 'accessible': True}
0O3uvYkHrEGhhl686jxS1D {'name': '126', 'type': 'Habitacion', 'level': 'Nivel 0', 'x': 18.0863181554455, 'y': 2.8503277388517874, 'z': 0.0, 'accessible': True}
0O3uvYkHrEGhhl686jxS0M {'name': '128', 'type': 'Habitacion', 'level': 'Nivel 2', 'x': 18.56863392082

## 6. Función de visualización 3D con Plotly

Esta celda define una visualización más legible que la del script original.

Mejoras principales:

- planta base en gris suave
- conexiones en naranja
- nodos diferenciados por tipo
- menú desplegable para mostrar cada nivel por separado


In [13]:
import os
import fiona
import geopandas as gpd
import plotly.graph_objects as go

def visualizar_grafo_plotly(gpkg_filename):
    if not os.path.exists(gpkg_filename):
        raise FileNotFoundError(f"No se encuentra el archivo {gpkg_filename}")

    gdf_nodos = gpd.read_file(gpkg_filename, layer="nodos")
    layers = fiona.listlayers(gpkg_filename)

    gdf_caminos = (
        gpd.read_file(gpkg_filename, layer="caminos")
        if "caminos" in layers else gpd.GeoDataFrame()
    )
    gdf_plantas = (
        gpd.read_file(gpkg_filename, layer="plantas_base")
        if "plantas_base" in layers else gpd.GeoDataFrame()
    )

    if "levels" not in gdf_caminos.columns:
        gdf_caminos["levels"] = ""

    niveles = sorted(gdf_nodos["level"].dropna().unique().tolist())

    styles = {
        "Habitacion": {"color": "#1f77b4", "symbol": "circle", "size": 5},
        "Suelo":      {"color": "#e377c2", "symbol": "circle-open", "size": 4},
        "Puerta":     {"color": "#2ca02c", "symbol": "diamond", "size": 4},
        "Escalera":   {"color": "#d62728", "symbol": "square", "size": 5},
        "Ascensor":   {"color": "#9467bd", "symbol": "x", "size": 6},
        "Rampa":      {"color": "#8c564b", "symbol": "cross", "size": 5},
    }

    data_traces = []
    trace_indices_by_level = {lvl: [] for lvl in niveles}
    current_idx = 0

    for nivel in niveles:
        # 1) Planta base simplificada
        if not gdf_plantas.empty:
            base_subset = gdf_plantas[gdf_plantas["level"] == nivel]
            bx, by, bz = [], [], []
            for line in base_subset.geometry:
                coords = list(line.coords)
                for i in range(len(coords) - 1):
                    p1, p2 = coords[i], coords[i + 1]
                    bx.extend([p1[0], p2[0], None])
                    by.extend([p1[1], p2[1], None])
                    bz.extend([p1[2], p2[2], None])

            trace_base = go.Scatter3d(
                x=bx, y=by, z=bz,
                mode="lines",
                line=dict(color="black", width=2),
                hoverinfo="none",
                name=f"Planta base ({nivel})",
                visible=True
            )
            data_traces.append(trace_base)
            trace_indices_by_level[nivel].append(current_idx)
            current_idx += 1

        # 2) Caminos
        edges_subset = (
            gdf_caminos[gdf_caminos["levels"].astype(str).str.contains(str(nivel), na=False)]
            if not gdf_caminos.empty else gpd.GeoDataFrame()
        )
        ex, ey, ez = [], [], []
        for line in edges_subset.geometry:
            coords = list(line.coords)
            for i in range(len(coords) - 1):
                p1, p2 = coords[i], coords[i + 1]
                ex.extend([p1[0], p2[0], None])
                ey.extend([p1[1], p2[1], None])
                ez.extend([p1[2], p2[2], None])

        trace_edges = go.Scatter3d(
            x=ex, y=ey, z=ez,
            mode="lines",
            line=dict(color="#ff7f0e", width=5),
            hoverinfo="none",
            name=f"Conexiones ({nivel})",
            visible=True
        )
        data_traces.append(trace_edges)
        trace_indices_by_level[nivel].append(current_idx)
        current_idx += 1

        # 3) Nodos
        nodos_subset = gdf_nodos[gdf_nodos["level"] == nivel]
        for node_type in nodos_subset["tipo"].unique():
            subset = nodos_subset[nodos_subset["tipo"] == node_type]
            style = styles.get(node_type, {"color": "gray", "symbol": "circle", "size": 5})

            trace_node = go.Scatter3d(
                x=subset.geometry.x,
                y=subset.geometry.y,
                z=subset.geometry.z,
                mode="markers",
                marker=dict(
                    size=style["size"],
                    color=style["color"],
                    symbol=style["symbol"],
                    opacity=0.92
                ),
                text=(
                    subset["tipo"].astype(str)
                    + "<br>ID: " + subset["id"].astype(str)
                    + "<br>Nivel: " + subset["level"].astype(str)
                ),
                name=f"{node_type} ({nivel})",
                visible=True
            )
            data_traces.append(trace_node)
            trace_indices_by_level[nivel].append(current_idx)
            current_idx += 1

    buttons = [
        dict(
            label="Mostrar todos",
            method="update",
            args=[{"visible": [True] * len(data_traces)}, {"title": "Grafo completo"}]
        )
    ]

    for nivel in niveles:
        visibility = [False] * len(data_traces)
        for idx in trace_indices_by_level[nivel]:
            visibility[idx] = True
        buttons.append(
            dict(
                label=f"Ver {nivel}",
                method="update",
                args=[{"visible": visibility}, {"title": f"Vista: {nivel}"}]
            )
        )

    layout = go.Layout(
        template=None,
        title=dict(text="Grafo de accesibilidad IFC", font=dict(color="black", size=16), y=0.96, x=0.5),
        font=dict(family="Arial", size=12, color="black"),
        updatemenus=[
            dict(
                type="dropdown",
                active=0,
                buttons=buttons,
                direction="down",
                x=1.0, y=1.12,
                xanchor="right", yanchor="top",
                bgcolor="#f2f2f2",
                font=dict(color="black"),
                bordercolor="gray",
                borderwidth=1
            )
        ],
        scene=dict(
            xaxis=dict(title="X", color="black", showbackground=False, tickfont=dict(color="black")),
            yaxis=dict(title="Y", color="black", showbackground=False, tickfont=dict(color="black")),
            zaxis=dict(title="Z", color="black", showbackground=False, tickfont=dict(color="black")),
            aspectmode="data",
            camera=dict(projection=dict(type="orthographic"))
        ),
        paper_bgcolor="white",
        plot_bgcolor="#f0f0f0", # Color de fondo del gráfico a gris claro
        legend=dict(
            yanchor="top", y=0.92,
            xanchor="left", x=0.01,
            font=dict(color="black"),
            bgcolor="#f2f2f2",
            bordercolor="gray",
            borderwidth=1
        ),
        margin=dict(t=90, b=0, l=0, r=0)
    )

    fig = go.Figure(data=data_traces, layout=layout)
    fig.show()

## 7. Mostrar el grafo y descargar el resultado

La última celda hace dos cosas:

1. abre la visualización 3D interactiva
2. descarga el GeoPackage generado

Ese archivo lo puedes abrir después en QGIS o seguir procesándolo en Python.


In [14]:
# Visualizar el grafo y descargar el GeoPackage resultante
visualizar_grafo_plotly(gpkg_filename)

from google.colab import files
files.download(gpkg_filename)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Observaciones técnicas importantes

- **Los recorridos ortogonales** ahora siguen la orientación del muro de la puerta cuando esa relación puede inferirse en IFC.
- **Los ascensores** ya no dependen de textos libres en `Name` u `ObjectType`.
- **Las rampas** se consideran accesibles por defecto.
- **La planta base** es una simplificación para mejorar la lectura del grafo, no un dibujo arquitectónico exhaustivo.
- **El GeoPackage** se exporta sin forzar un CRS geográfico, porque la mayoría de IFC trabajan en coordenadas locales de ingeniería.
